# NB20 The security budget — Chapter 20

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB20-1: Bitcoin's subsidy schedule, a port of Core's GetBlockSubsidy in the same integer arithmetic (for study)
COIN = 100_000_000                      # 1 BTC = 1e8 satoshis (consensus/amount.h)
HALVING = 210_000                       # nSubsidyHalvingInterval (chainparams.cpp)

def block_subsidy_sat(height: int) -> int:
    halvings = height // HALVING
    if halvings >= 64:                  # validation.cpp: 0 after 64 or more halvings (guards the shift width)
        return 0
    return (50 * COIN) >> halvings      # halve 50 BTC `halvings` times; an integer bit shift, so anything below 1 satoshi is discarded

print(f"{'halvings':>8} {'start height':>12} {'block subsidy (BTC)':>20} {'cumulative issued by end (BTC)':>30}")
cum = 0
for h in range(0, 36):
    start = h * HALVING
    sub = block_subsidy_sat(start)
    cum += sub * HALVING
    if h <= 6 or sub == 0 or h in (10, 20, 32, 33, 34):
        print(f"{h:>8} {start:>12,} {sub / COIN:>20.8f} {cum / COIN:>30.8f}")
    if sub == 0:
        break
total = sum(block_subsidy_sat(h * HALVING) * HALVING for h in range(0, 64))
print(f"\nSum of the subsidy in integer arithmetic: {total:,} satoshis = {total / COIN:.8f} BTC ({21_000_000 - total / COIN:.8f} BTC short of 21,000,000)")
tip = 972_958                           # chain-tip height observed 2026-10-24 UTC (data/hashrate_2026-10-25.json)
print(f"Height on the reference date {tip:,}: after {tip // HALVING} halvings, block subsidy {block_subsidy_sat(tip) / COIN} BTC, next halving at height {(tip // HALVING + 1) * HALVING:,} (about {((tip // HALVING + 1) * HALVING - tip) * 10 / 60 / 24:.0f} days away, assuming 10 minutes per block)")

halvings start height  block subsidy (BTC) cumulative issued by end (BTC)
       0            0          50.00000000              10500000.00000000
       1      210,000          25.00000000              15750000.00000000
       2      420,000          12.50000000              18375000.00000000
       3      630,000           6.25000000              19687500.00000000
       4      840,000           3.12500000              20343750.00000000
       5    1,050,000           1.56250000              20671875.00000000
       6    1,260,000           0.78125000              20835937.50000000
      10    2,100,000           0.04882812              20989746.09270000
      20    4,200,000           0.00004768              20999989.97250000
      32    6,720,000           0.00000001              20999999.97690000
      33    6,930,000           0.00000000              20999999.97690000

Sum of the subsidy in integer arithmetic: 2,099,999,997,690,000 satoshis = 20999999.97690000 BTC (0.02310000 BT

In [ ]:
# NB20-2: Budish's Proposition 3, the expected duration t(A, e, k) of a double-spending attack and the gross cost A·t in units of honest block rewards (for study)
# The only inputs are the attacker multiple A (share A/(1+A)) and the block count k+e. No coin price, no fiat unit price.
from math import lgamma, log, exp

def attack_duration(A: float, ke: int) -> float:
    """Equation (6). Factorials via log-gamma, to avoid overflow at large ke. Unit: honest block time (≈10 minutes)."""
    t = float(ke)
    for i in range(0, ke + 1):
        log_comb = lgamma(2 * ke - i) - lgamma(ke - i + 1) - lgamma(ke)   # (2ke−1−i)! / ((ke−i)! (ke−1)!)
        log_prob = (ke - i) * log(A / (1 + A)) + ke * log(1 / (1 + A))
        t += (i + 1) / (A - 1) * exp(log_comb + log_prob)
    return t

rows = [1.05, 1.10, 1.20, 1.33, 1.50, 2.00, 5.00]
cols = [1, 6, 12, 36, 144]
print("Expected duration t (unit: honest block time). Columns = k+e (double-spent blocks + escrow)")
print(f"{'A (share)':>14}" + "".join(f"{c:>10}" for c in cols))
for A in rows:
    print(f"{A:>5.2f} ({A/(1+A):4.0%})" + "".join(f"{attack_duration(A, c):>10.2f}" for c in cols))
print("\nGross cost A·t (unit: honest trust-support cost per block N*c = block reward p_block)")
print(f"{'A (share)':>14}" + "".join(f"{c:>10}" for c in cols))
for A in rows:
    print(f"{A:>5.2f} ({A/(1+A):4.0%})" + "".join(f"{A * attack_duration(A, c):>10.2f}" for c in cols))

# The attacker picks the A that minimizes A·t. Grid search at k+e = 12 (the paper's base case)
ke = 12
best = min(((A / 100) * attack_duration(A / 100, ke), A / 100) for A in range(101, 600))
print(f"\nk+e = {ke}: the A that minimizes A·t is A* ≈ {best[1]:.2f} (share {best[1]/(1+best[1]):.0%}), minimum A*·t(A*) ≈ {best[0]:.2f} block rewards")
print(f"→ equation (3): p_block > V_attack / {best[0]:.1f}, i.e., the reward per block must exceed {1/best[0]:.2%} of the attack value")

Expected duration t (unit: honest block time). Columns = k+e (double-spent blocks + escrow)
     A (share)         1         6        12        36       144
 1.05 ( 51%)     25.51     42.43     56.59     97.91    229.61
 1.10 ( 52%)     13.02     22.88     31.73     60.14    167.31
 1.20 ( 55%)      6.79     13.24     19.69     43.10    146.79
 1.33 ( 57%)      4.34      9.60     15.35     38.04    144.18
 1.50 ( 60%)      3.08      7.84     13.43     36.47    144.00
 2.00 ( 67%)      1.89      6.45     12.20     36.01    144.00
 5.00 ( 83%)      1.12      6.00     12.00     36.00    144.00

Gross cost A·t (unit: honest trust-support cost per block N*c = block reward p_block)
     A (share)         1         6        12        36       144
 1.05 ( 51%)     26.78     44.56     59.42    102.81    241.09
 1.10 ( 52%)     14.32     25.16     34.90     66.16    184.04
 1.20 ( 55%)      8.14     15.89     23.63     51.72    176.15
 1.33 ( 57%)      5.78     12.77     20.41     50.59    191.7

In [ ]:
# NB20-3: attack-cost calculator. Inputs: hash rate (snapshot), attacker multiple, unit price, duration. The coin price is not an input (for study)
import json, os

# Hash-rate snapshot for the reference date (data bundled; if the file is missing, a constant with the same value is used, so it runs on Colab without the file)
try:
    with open(os.path.join("data", "hashrate_2026-10-25.json"), encoding="utf-8") as f:
        snap = json.load(f)
    H_network = float(snap["estimate"]["hashrate_H_per_s"])
except FileNotFoundError:
    H_network = 1.006294e21                 # H/s, estimated from difficulty 140,577,680,051,919.6 as D·2^32/600 (600 seconds assumed)
print(f"Estimated hash rate on the reference date H = {H_network:.3e} H/s = {H_network/1e18:,.0f} EH/s (2026-10-24 UTC, estimate assuming 600 seconds; about 7% below the comparison value)")

def attack_cost(A, ke, c_per_hash_second=None, p_block=None):
    """Gross cost. Given c_per_hash_second, in "unit-price units × hash-seconds"; given p_block, in "block-reward units" via equation (1)."""
    t_blocks = attack_duration(A, ke)
    seconds = t_blocks * 600
    out = {"t_blocks": t_blocks, "hours": seconds / 3600, "attacker_hashrate_hs": A * H_network}
    if c_per_hash_second is not None:
        out["cost_in_price_units"] = A * H_network * c_per_hash_second * seconds
    if p_block is not None:
        out["cost_in_block_rewards"] = A * t_blocks            # equation (1): N*c = p_block
        out["cost_in_coin_of_reward"] = A * t_blocks * p_block
    return out

p_block_subsidy_only = 3.125                 # subsidy on the reference date (BTC). The fee portion F varies from block to block, so this is a lower bound with it set to 0
print(f"\n(a) In block-reward units: a lower bound with fees set to 0, p_block = {p_block_subsidy_only} BTC (subsidy only)")
print(f"{'share':>6} {'A':>5} {'k+e':>4} {'expected blocks':>15} {'time (h)':>8} {'block rewards':>13} {'BTC equiv. (subsidy only)':>25}")
for q in [0.55, 0.60, 0.67]:
    A = q / (1 - q)
    for ke in [6, 12, 144]:
        r = attack_cost(A, ke, p_block=p_block_subsidy_only)
        print(f"{q:>6.0%} {A:>5.2f} {ke:>4} {r['t_blocks']:>15.1f} {r['hours']:>8.1f} {r['cost_in_block_rewards']:>13.1f} {r['cost_in_coin_of_reward']:>25.1f}")

print("\n(b) In the user's unit-price units: c is not the cost per 1 hash but 'the cost of running 1 H/s for 1 second'. The value below is a hypothetical unit price that only fills the slot.")
c_demo = 1e-19                                # hypothetical unit price (no unit). The user enters actual electricity and equipment unit prices
for q in [0.55, 0.60]:
    A = q / (1 - q)
    r = attack_cost(A, 12, c_per_hash_second=c_demo)
    print(f"  share {q:.0%}, k+e=12: attacker hash rate {r['attacker_hashrate_hs']:.2e} H/s × {r['hours']:.1f} hours × unit price → cost {r['cost_in_price_units']:,.0f} (unit-price units)")
print("\nAt the same share and the same duration, the cost is proportional to the unit price c. Enter half the unit price and the cost halves: what this calculator tells you is a proportion, not a level.")

Estimated hash rate on the reference date H = 1.006e+21 H/s = 1,006 EH/s (2026-10-24 UTC, estimate assuming 600 seconds; about 7% below the comparison value)

(a) In block-reward units: a lower bound with fees set to 0, p_block = 3.125 BTC (subsidy only)
 share     A  k+e expected blocks time (h) block rewards BTC equiv. (subsidy only)
   55%  1.22    6            12.3      2.0          15.0                      47.0
   55%  1.22   12            18.5      3.1          22.7                      70.8
   55%  1.22  144           145.8     24.3         178.2                     556.7
   60%  1.50    6             7.8      1.3          11.8                      36.7
   60%  1.50   12            13.4      2.2          20.1                      62.9
   60%  1.50  144           144.0     24.0         216.0                     675.0
   67%  2.03    6             6.4      1.1          13.0                      40.7
   67%  2.03   12            12.2      2.0          24.7                      77.